<a href="https://colab.research.google.com/github/Feznix/Stats-554/blob/Homework-5/Homework6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Name: Heather Secrest

Date: 05/11/2026

Course: ST 554 (601)

Homework 6

#**PART 1**

##**Question 1**

In [11]:
#bring in the module
import sqlite3
import pandas as pd
import numpy as np
#make the connection to the file. My file is in the main folder on the left
MLB = sqlite3.connect("lahman_1871-2022.sqlite")

I used the `pd.read_sql()` as a method of viewing the schema as a dataframe. I basically took this from what I did on HW 5.

In [6]:
#use the pd.read_sql() to return a dataframe
# SQL selects all columns from the schema where the type is a table
pd.read_sql('SELECT * FROM sqlite_schema \
            WHERE type = "table"',MLB)

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


##**Question 2**

I started by creating a new table with all of the desired columns and the names I wanted for them. I used pd.read_sql to check if I had created the table and I had. I used the sql CREATE TABLE IF NOT EXISTS. The process of querying came from the lecture. I designated the columns with numbers as REAL instead of INTEGER for ease of doing percentage calculations with them later (I know we aren't but that is the most likely stat to do with the numbers).

In [51]:
#create a 'cursor' object from our connection
cursor = MLB.cursor()

#SQL query to create a new table in the data base

new_table = """
          CREATE TABLE IF NOT EXISTS HallofFamePitchers(
            playerID TEXT,
            Total_GS REAL,
            Total_G REAL,
            Total_W REAL,
            Total_L REAL,
            Total_IPOuts REAL,
            Total_CG REAL,
            Total_SHO REAL,
           Total_SV REAL)
        """

#execute the SQL query on the database!
cursor.execute(new_table)

#The information for the query is stored in memory. We use the fetchall() method to actually return the information
result = cursor.fetchall()

#finall we close the connection the cursor made with that query
cursor.close()

Now that I had a new table I needed to populate it with information from other tables. I did this using INSERT INTO. I used SELECT to get the data I wanted from the HallOfFame table and the Pitching table. I made sure to order them to match my column orders that I added above. I then used INNER JOIN to get only the data that was for hall of fame pitchers and GROUP BY so that the SUM() function used to get my totals would work properly.

In [52]:
#create a 'cursor' object from our connection
cursor = MLB.cursor()

#SQL add data to a table in the data base from exsisting data and tables

insert_data = """
              INSERT INTO HallofFamePitchers(playerID,
               Total_GS, Total_G,
               Total_W, Total_L,
               Total_IPOuts, Total_CG,
               Total_SHO, Total_SV)
              SELECT h.playerID,
               SUM(p.GS), SUM(p.G),
               SUM(p.W), SUM(p.L),
                SUM(p.IPouts), SUM(p.CG),
               SUM(p.SHO), SUM(p.SV)
              FROM HallofFame AS h
               INNER JOIN Pitching AS p
                ON h.playerID = p.playerID
              GROUP BY h.playerID"""

#execute the SQL query on the database!
cursor.executescript(insert_data)

#The information for the query is stored in memory. We use the fetchall() method to actually return the information
result = cursor.fetchall()

#finall we close the connection the cursor made with that query
cursor.close()

I had to add the DROP TABLE thing because I messed the INSERT DATA up a couple of times (I had an extra ; that messed stuff up and once my group by didn't work right). By dropping the table and starting from my initial create table code I could delete the mess ups.

In [50]:
#create a 'cursor' object from our connection
cursor = MLB.cursor()

#SQL query to create a new table in the data base

drop_table = """ DROP TABLE HallofFamePitchers;
        """

#execute the SQL query on the database!
cursor.execute(drop_table)

#The information for the query is stored in memory. We use the fetchall() method to actually return the information
result = cursor.fetchall()

#finall we close the connection the cursor made with that query
cursor.close()

Used to confirm what data was added to my table.

In [53]:
pd.read_sql('''SELECT *
              FROM HallofFamePitchers''',MLB)

,playerID,Total_GS,Total_G,Total_W,Total_L,Total_IPOuts,Total_CG,Total_SHO,Total_SV
0,abbotji01,254.0,263.0,87.0,108.0,5022.0,31.0,6.0,0.0
1,adamsba01,5325.0,7230.0,2910.0,2100.0,134790.0,3090.0,660.0,225.0
2,aguilri01,89.0,732.0,86.0,81.0,3874.0,10.0,0.0,318.0
3,akerja01,0.0,495.0,47.0,45.0,2238.0,0.0,0.0,123.0
4,alexado01,464.0,561.0,194.0,174.0,10103.0,98.0,18.0,3.0
...,...,...,...,...,...,...,...,...,...
522,zachrpa01,154.0,293.0,69.0,67.0,3532.0,29.0,7.0,3.0
523,zahnge01,270.0,304.0,111.0,109.0,5547.0,79.0,20.0,1.0
524,zambrca01,302.0,354.0,132.0,91.0,5877.0,10.0,5.0,0.0
525,zeileto01,0.0,2.0,0.0,0.0,6.0,0.0,0.0,0.0


Used to confirm that the data I got above matched the data I would get by using the SQL with pd.read_sql().

In [39]:
pd.read_sql( '''SELECT h.playerID,
              SUM(p.GS), SUM(p.G),
              SUM(p.W), SUM(p.L),
              SUM(p.IPouts), SUM(p.CG),
              SUM(p.SHO), SUM(p.SV)
              FROM HallofFame AS h
              INNER JOIN Pitching AS p
              ON h.playerID = p.playerID
              GROUP BY h.playerID''',MLB)

,playerID,SUM(p.GS),SUM(p.G),SUM(p.W),SUM(p.L),SUM(p.IPouts),SUM(p.CG),SUM(p.SHO),SUM(p.SV)
0,abbotji01,254,263,87,108,5022,31,6,0
1,adamsba01,5325,7230,2910,2100,134790,3090,660,225
2,aguilri01,89,732,86,81,3874,10,0,318
3,akerja01,0,495,47,45,2238,0,0,123
4,alexado01,464,561,194,174,10103,98,18,3
...,...,...,...,...,...,...,...,...,...
522,zachrpa01,154,293,69,67,3532,29,7,3
523,zahnge01,270,304,111,109,5547,79,20,1
524,zambrca01,302,354,132,91,5877,10,5,0
525,zeileto01,0,2,0,0,6,0,0,0


##**Question 3**

I used basically the same code from Question 2 to create my new table. I named it something different and included all of the table columns that I wanted to include.

In [67]:
#create a 'cursor' object from our connection
cursor = MLB.cursor()

#SQL query to create a new table in the data base

new_table2 = """
          CREATE TABLE IF NOT EXISTS HallofFamePitchers_Batting(
            playerID TEXT,Total_AB REAL,
            Total_R REAL, Total_H REAL,
            Total_HR REAL, Total_RBI REAL,
            Total_BB REAL, Total_SO REAL)
        """

#execute the SQL query on the database!
cursor.execute(new_table2)

#The information for the query is stored in memory. We use the fetchall() method to actually return the information
result = cursor.fetchall()

#finall we close the connection the cursor made with that query
cursor.close()

Again, the process here basically matched with question 2. A key difference is that I used the table created in Question 2 to get my playerIDs for my INNER JOIN beccause I wanted the pitchers in the hall of fame spesifically. Using my new table meant that I was sure the playerIDs of this table match the one I just made.

In [68]:
#create a 'cursor' object from our connection
cursor = MLB.cursor()

#SQL add data to a table in the data base from exsisting data and tables

insert_data = """
              INSERT INTO HallofFamePitchers_Batting( playerID,
                 Total_AB, Total_R,
                 Total_H, Total_HR,
                 Total_RBI, Total_BB,
                 Total_SO)
              SELECT h.playerID,
                 SUM(b.AB),SUM(b.R),
                 SUM(b.H),SUM(b.HR),
                 SUM(b.RBI),SUM(b.BB),
                 SUM(b.SO)
              FROM HallofFamePitchers AS h
              INNER JOIN Batting AS b
              ON h.playerID = b.playerID
              GROUP BY h.playerID"""

#execute the SQL query on the database!
cursor.executescript(insert_data)

#The information for the query is stored in memory. We use the fetchall() method to actually return the information
result = cursor.fetchall()

#finall we close the connection the cursor made with that query
cursor.close()

Drop table used to reset if needed.

In [66]:
#create a 'cursor' object from our connection
cursor = MLB.cursor()

#SQL query to create a new table in the data base

drop_table = """ DROP TABLE HallofFamePitchers_Batting;
        """

#execute the SQL query on the database!
cursor.execute(drop_table)

#The information for the query is stored in memory. We use the fetchall() method to actually return the information
result = cursor.fetchall()

#finall we close the connection the cursor made with that query
cursor.close()

Here is my pd.read_sql() to check the results of the code above.

In [62]:
pd.read_sql('''SELECT *
              FROM HallofFamePitchers_Batting''',MLB)

,playerID,Total_AB,Total_R,Total_H,Total_HR,Total_RBI,Total_BB,Total_SO
0,abbotji01,21.0,0.0,2.0,0.0,3.0,0.0,10.0
1,adamsba01,1019.0,79.0,216.0,3.0,75.0,53.0,194.0
2,aguilri01,139.0,12.0,28.0,3.0,11.0,6.0,37.0
3,akerja01,92.0,3.0,7.0,0.0,4.0,1.0,51.0
4,alexado01,265.0,19.0,44.0,0.0,17.0,9.0,77.0
...,...,...,...,...,...,...,...,...
522,zachrpa01,318.0,9.0,36.0,0.0,6.0,14.0,128.0
523,zahnge01,43.0,3.0,6.0,0.0,1.0,0.0,9.0
524,zambrca01,693.0,75.0,165.0,24.0,71.0,10.0,240.0
525,zeileto01,7573.0,986.0,2004.0,253.0,1110.0,945.0,1279.0


And here is the same table, but instead called from other tables as a temporary table to confirm that the table above is what I wanted.

In [64]:
pd.read_sql('''SELECT h.playerID,
                 SUM(b.AB),SUM(b.R),
                 SUM(b.H),SUM(b.HR),
                 SUM(b.RBI),SUM(b.BB),
                 SUM(b.SO)
              FROM HallofFamePitchers AS h
              INNER JOIN Batting AS b
              ON h.playerID = b.playerID
              GROUP BY h.playerID''',MLB)

,playerID,SUM(b.AB),SUM(b.R),SUM(b.H),SUM(b.HR),SUM(b.RBI),SUM(b.BB),SUM(b.SO)
0,abbotji01,21,0,2,0,3.0,0,10.0
1,adamsba01,1019,79,216,3,75.0,53,194.0
2,aguilri01,139,12,28,3,11.0,6,37.0
3,akerja01,92,3,7,0,4.0,1,51.0
4,alexado01,265,19,44,0,17.0,9,77.0
...,...,...,...,...,...,...,...,...
522,zachrpa01,318,9,36,0,6.0,14,128.0
523,zahnge01,43,3,6,0,1.0,0,9.0
524,zambrca01,693,75,165,24,71.0,10,240.0
525,zeileto01,7573,986,2004,253,1110.0,945,1279.0


##**Question4**

Honestly, I tried to do this using SQL at first, but I got confused and felt like it would take way too long to do. So, I decided to use pd.merge(). I specified that I wanted an outer join because it made the most sense.

In [73]:
Batting=pd.read_sql('''SELECT *
              FROM HallofFamePitchers_Batting''',MLB)
Pitching=pd.read_sql('''SELECT *
              FROM HallofFamePitchers''',MLB)
Combined=pd.merge(Batting,Pitching,'outer')

In [74]:
Combined

,playerID,Total_AB,Total_R,Total_H,Total_HR,Total_RBI,Total_BB,Total_SO,Total_GS,Total_G,Total_W,Total_L,Total_IPOuts,Total_CG,Total_SHO,Total_SV
0,abbotji01,21.0,0.0,2.0,0.0,3.0,0.0,10.0,254.0,263.0,87.0,108.0,5022.0,31.0,6.0,0.0
1,adamsba01,1019.0,79.0,216.0,3.0,75.0,53.0,194.0,5325.0,7230.0,2910.0,2100.0,134790.0,3090.0,660.0,225.0
2,aguilri01,139.0,12.0,28.0,3.0,11.0,6.0,37.0,89.0,732.0,86.0,81.0,3874.0,10.0,0.0,318.0
3,akerja01,92.0,3.0,7.0,0.0,4.0,1.0,51.0,0.0,495.0,47.0,45.0,2238.0,0.0,0.0,123.0
4,alexado01,265.0,19.0,44.0,0.0,17.0,9.0,77.0,464.0,561.0,194.0,174.0,10103.0,98.0,18.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,zachrpa01,318.0,9.0,36.0,0.0,6.0,14.0,128.0,154.0,293.0,69.0,67.0,3532.0,29.0,7.0,3.0
523,zahnge01,43.0,3.0,6.0,0.0,1.0,0.0,9.0,270.0,304.0,111.0,109.0,5547.0,79.0,20.0,1.0
524,zambrca01,693.0,75.0,165.0,24.0,71.0,10.0,240.0,302.0,354.0,132.0,91.0,5877.0,10.0,5.0,0.0
525,zeileto01,7573.0,986.0,2004.0,253.0,1110.0,945.0,1279.0,0.0,2.0,0.0,0.0,6.0,0.0,0.0,0.0
